# 03 · Model Design: Model 1 Baseline
**Proyek:** Prediksi Harga Harian Komoditas Pangan (SDG 2, Target 2.c) · **Minggu 3** [M] Model Design: Baseline & Advanced

Notebook ini mengerjakan **Model 1 (Baseline)**: Bab 6.1.1 sampai 6.1.5.

| Bagian | Isi | Bab |
|---|---|---|
| 1 | Memuat data hasil `02_data_preparation` | - |
| 2 | Deskripsi model, hypothesis space, inductive bias (teks draf) | 6.1.1 – 6.1.2 |
| 3 | Implementasi: naif, Linear Regression, Ridge | 6.1.3 |
| 4 | Pemilihan hyperparameter dan regularisasi | 6.1.4 |
| 5 | Hasil awal di validation set dan analisis kesalahan | 6.1.5 |
| 6 | Menyimpan hasil untuk perbandingan dengan Model 2 | tugas 5 |

**Aturan yang dipegang notebook ini:**
1. **Test set tidak dimuat.** Hanya train dan validation.
2. Hyperparameter dan pemilihan model memakai **cross-validation berurutan (`TimeSeriesSplit`) pada data latih**. Validation set hanya untuk melaporkan hasil.
3. Semua galat dihitung dalam satuan Rp/kg, termasuk untuk target selisih harga.

## 0. Setup dan Konfigurasi

In [ ]:
import json, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
random.seed(SEED); np.random.seed(SEED)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (11, 4)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data" / "processed"
MODEL_DIR = ROOT / "models"
REPORT_DIR = ROOT / "reports"
FIG_DIR = REPORT_DIR / "figures"
for d in (MODEL_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ===== KONFIGURASI =====
DATE_COL = "tanggal"            # sesuaikan dengan notebook 02
TARGET_DATE_COL = "target_date"
ALPHA_GRID = [0.001, 0.01, 0.1, 1, 10, 100, 1000]   # kandidat kekuatan regularisasi Ridge
N_SPLITS = 5                    # jumlah lipatan TimeSeriesSplit

## 1. Memuat Data
Berkas berikut dihasilkan oleh `02_data_preparation`. Fitur ber-scaling dipakai oleh model linear, sedangkan kolom `y` dan `harga_t` asli (Rp/kg) dipakai untuk menghitung galat.

In [ ]:
def baca(nama):
    path = DATA_DIR / nama
    if not path.exists():
        raise FileNotFoundError(f"{path} belum ada. Jalankan notebook 02_data_preparation terlebih dahulu.")
    return pd.read_csv(path)

tr, va = baca("features_train.csv"), baca("features_val.csv")
tr_s, va_s = baca("features_scaled_train.csv"), baca("features_scaled_val.csv")

NON_FEATURES = [DATE_COL, TARGET_DATE_COL, "y", "split"]
FEATURE_COLS = [c for c in tr_s.columns if c not in NON_FEATURES]

# pemeriksaan kesesuaian data
assert (tr["split"] == "train").all() and (va["split"] == "val").all(), "Berkas split tidak sesuai"
assert pd.to_datetime(tr[TARGET_DATE_COL]).max() < pd.to_datetime(va[TARGET_DATE_COL]).min(), "Train dan val tumpang tindih"
assert (tr[DATE_COL].values == tr_s[DATE_COL].values).all() and (va[DATE_COL].values == va_s[DATE_COL].values).all()
assert "harga_t" in FEATURE_COLS

X_tr, X_va = tr_s[FEATURE_COLS].to_numpy(), va_s[FEATURE_COLS].to_numpy()
y_tr, y_va = tr["y"].to_numpy(), va["y"].to_numpy()
h_tr, h_va = tr["harga_t"].to_numpy(), va["harga_t"].to_numpy()   # harga hari ini (Rp/kg), dipakai juga untuk prediksi naif

print(f"Train: {len(tr)} sampel | Val: {len(va)} sampel | Jumlah fitur: {len(FEATURE_COLS)}")
print("Fitur:", FEATURE_COLS)

## 2. Deskripsi Model dan Hypothesis Space (Bab 6.1.1 – 6.1.2)
Teks di bawah adalah **draf** untuk laporan. Sesuaikan angka dengan hasil Anda, dan ubah nama model jika model terpilih di Bagian 4 berbeda.

### 6.1.1 Deskripsi Model
**Nama Model:** Ridge Regression (Linear Regression dengan regularisasi L2), dibandingkan dengan Linear Regression biasa dan prediksi naif (harga hari ini sebagai prediksi harga besok).

**Teori singkat:** model memprediksi target sebagai kombinasi linear fitur, ŷ = w·x + b. Parameter dipilih dengan meminimalkan jumlah kuadrat galat ditambah penalti α‖w‖². Penalti ini menyusutkan koefisien sehingga lebih stabil ketika fitur saling berkorelasi.

**Alasan pemilihan:** sederhana, cepat, dan mudah dijelaskan sebagai titik acuan. Pada data ini, autokorelasi harga sangat tinggi (sekitar 0,995 pada lag 1 dan 0,921 pada lag 7), sehingga fitur lag dan rolling saling berkorelasi kuat dan regularisasi diperlukan. Harga juga sangat persisten, sehingga selain target harga langsung dicoba target **selisih harga** (`y − harga_t`) agar model tidak sekadar meniru harga hari ini.

### 6.1.2 Hypothesis Space & Inductive Bias
- **Model family / arsitektur:** model linear dengan regularisasi L2.
- **Hypothesis space:** himpunan fungsi linear dari fitur (lag, statistik bergulir, kalender) ke target (harga atau selisih harga), dengan besar koefisien dibatasi oleh α.
- **Inductive bias / asumsi:** hubungan fitur dan target linear dan aditif, pengaruh tiap fitur konstan sepanjang waktu, tidak ada interaksi antar-fitur, dan koefisien kecil lebih disukai.
- **Implikasi terhadap dataset:** asumsi cukup cocok untuk dinamika harian yang persisten, tetapi kurang cocok untuk lonjakan dan siklus non-linear serta efek hari raya yang bergantung konteks. Kesalahan besar diharapkan muncul di periode puncak, dan ini menjadi alasan memakai Model 2 dan Model 3.

## 3. Implementasi (Bab 6.1.3)
Empat konfigurasi dibandingkan terhadap baseline naif:

| Konfigurasi | Target yang dipelajari | Prediksi akhir |
|---|---|---|
| Linear Regression (level) | harga hari berikutnya | langsung |
| Linear Regression (selisih) | `y − harga_t` | `harga_t` + prediksi |
| Ridge (level) | harga hari berikutnya | langsung |
| Ridge (selisih) | `y − harga_t` | `harga_t` + prediksi |

In [ ]:
def metrik(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
            "MAPE_%": float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)}

def ke_target(y, h, mode):
    return y - h if mode == "selisih" else y

def ke_harga(pred, h, mode):
    return pred + h if mode == "selisih" else pred

tscv = TimeSeriesSplit(n_splits=N_SPLITS)

def cv_mae(buat_model, mode):
    """MAE rata-rata (Rp/kg) pada lipatan TimeSeriesSplit di DATA LATIH saja."""
    galat = []
    for a, b in tscv.split(X_tr):
        m = buat_model().fit(X_tr[a], ke_target(y_tr[a], h_tr[a], mode))
        p = ke_harga(m.predict(X_tr[b]), h_tr[b], mode)
        galat.append(mean_absolute_error(y_tr[b], p))
    return float(np.mean(galat))

# pembanding naif pada lipatan yang sama
cv_naif = float(np.mean([mean_absolute_error(y_tr[b], h_tr[b]) for _, b in tscv.split(X_tr)]))
print(f"MAE naif pada CV data latih: {cv_naif:,.0f}")

## 4. Optimization: Objective, Hyperparameter, dan Regularisasi (Bab 6.1.4)
**Objective:** meminimalkan jumlah kuadrat galat (+ penalti L2 untuk Ridge). Pemilihan `alpha` dan model memakai **MAE pada CV berurutan di data latih**, bukan validation set.

In [ ]:
baris = []
for mode in ("level", "selisih"):
    baris.append({"model": "LinReg", "target": mode, "alpha": np.nan,
                  "cv_MAE": cv_mae(lambda: LinearRegression(), mode)})
    for a in ALPHA_GRID:
        baris.append({"model": "Ridge", "target": mode, "alpha": a,
                      "cv_MAE": cv_mae(lambda a=a: Ridge(alpha=a), mode)})
cv_df = pd.DataFrame(baris)
display(cv_df.round(2))

fig, ax = plt.subplots()
for mode in ("level", "selisih"):
    s = cv_df[(cv_df.model == "Ridge") & (cv_df.target == mode)]
    ax.plot(s["alpha"], s["cv_MAE"], marker="o", label=f"Ridge ({mode})")
    lr = cv_df[(cv_df.model == "LinReg") & (cv_df.target == mode)]["cv_MAE"].iloc[0]
    ax.axhline(lr, ls="--", alpha=0.5, label=f"LinReg ({mode})")
ax.axhline(cv_naif, c="gray", ls=":", label="naif")
ax.set_xscale("log"); ax.set_xlabel("alpha"); ax.set_ylabel("MAE CV (Rp/kg)")
ax.set_title("Pemilihan alpha dengan TimeSeriesSplit (data latih)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG_DIR / "baseline_alpha_cv.png", dpi=150); plt.show()

**Catatan hyperparameter (isi):**
- `alpha` terbaik untuk Ridge (level) dan Ridge (selisih): [ ... ]
- Apakah regularisasi memperbaiki Linear Regression biasa? [ ... ]
- Apakah target selisih memperbaiki hasil dibanding target level? [ ... ]

## 5. Hasil Awal pada Validation Set (Bab 6.1.5)
Setiap konfigurasi dilatih ulang pada seluruh data latih dengan `alpha` terbaik dari CV, lalu dievaluasi **sekali** di validation set.

In [ ]:
final_models, preds, hasil = {}, {}, {}
hasil["Naif"] = {"cv_MAE": cv_naif, "alpha": np.nan, **metrik(y_va, h_va)}
preds["Naif"] = h_va

for (nama, mode), g in cv_df.groupby(["model", "target"]):
    terbaik = g.loc[g["cv_MAE"].idxmin()]
    mdl = LinearRegression() if nama == "LinReg" else Ridge(alpha=float(terbaik["alpha"]))
    mdl.fit(X_tr, ke_target(y_tr, h_tr, mode))
    key = f"{nama} ({mode})"
    final_models[key] = mdl
    preds[key] = ke_harga(mdl.predict(X_va), h_va, mode)
    hasil[key] = {"cv_MAE": float(terbaik["cv_MAE"]), "alpha": terbaik["alpha"], **metrik(y_va, preds[key])}

tabel = pd.DataFrame(hasil).T
tabel.index.name = "model"
display(tabel.sort_values("cv_MAE").round(2))

kandidat = [k for k in hasil if k != "Naif"]
BEST = min(kandidat, key=lambda k: hasil[k]["cv_MAE"])
print(f"\nModel 1 terpilih (MAE CV terendah di data latih): {BEST}")
if hasil[BEST]["MAE"] >= hasil["Naif"]["MAE"]:
    print("Catatan: MAE validation belum mengalahkan baseline naif. Ini temuan yang sah dan perlu dibahas di laporan.")
else:
    sel = 100 * (1 - hasil[BEST]["MAE"] / hasil["Naif"]["MAE"])
    print(f"MAE validation lebih rendah {sel:.1f}% dibanding baseline naif.")

In [ ]:
# Koefisien: bukti pengaruh multikolinearitas dan efek regularisasi
koef = {}
for k in ("LinReg (selisih)", "Ridge (selisih)", "LinReg (level)", "Ridge (level)"):
    koef[k] = pd.Series(final_models[k].coef_, index=FEATURE_COLS)
display(pd.DataFrame(koef).round(3))

lag_cols = [c for c in FEATURE_COLS if c.startswith(("harga_t", "lag_", "roll_mean"))]
print("Korelasi antar fitur harga (data latih):")
display(tr_s[lag_cols].corr().round(3))

**Catatan koefisien (isi):** Apakah koefisien Linear Regression biasa jauh lebih besar atau berlawanan tanda antar-lag dibanding Ridge? Fitur mana yang paling berpengaruh? [ ... ]

In [ ]:
# Plot prediksi vs aktual dan analisis kesalahan di validation set
tgl = pd.to_datetime(va[TARGET_DATE_COL])
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(tgl, y_va, label="aktual", lw=1.2)
axes[0].plot(tgl, preds[BEST], label=BEST, lw=1.2)
axes[0].plot(tgl, h_va, label="naif", lw=0.8, alpha=0.6)
axes[0].set_ylabel("Rp/kg"); axes[0].legend(); axes[0].set_title("Validation set: prediksi vs aktual")
axes[1].plot(tgl, y_va - preds[BEST], lw=0.9)
axes[1].axhline(0, c="gray", lw=0.8); axes[1].set_ylabel("galat (aktual - prediksi)")
plt.tight_layout(); plt.savefig(FIG_DIR / "baseline_val_prediksi.png", dpi=150); plt.show()

gal = va[[TARGET_DATE_COL, "y", "harga_t"]].assign(prediksi=preds[BEST])
gal["abs_galat"] = (gal["y"] - gal["prediksi"]).abs()
if "dekat_hari_raya" in va.columns:
    gal["dekat_hari_raya"] = va["dekat_hari_raya"]
print("10 galat terbesar di validation set:")
display(gal.sort_values("abs_galat", ascending=False).head(10).round(0))

**Hasil awal (isi untuk Bab 6.1.5):**
- Skor validation model terpilih: MAE [ ... ], RMSE [ ... ], MAPE [ ... ] %
- Dibanding baseline naif: [lebih baik / hampir sama / lebih buruk] sebesar [ ... ]
- Pola kesalahan: [apakah galat besar muncul pada periode lonjakan, saat harga berbalik arah, atau dekat hari raya?]
- Kesimpulan untuk Model 2: [apa yang diharapkan bisa dipelajari model non-linear yang tidak bisa dipelajari baseline?]

> Evaluasi akhir pada test set dijelaskan di Bab 7, bukan di notebook ini.

## 6. Menyimpan Hasil untuk Perbandingan dengan Model 2

In [ ]:
joblib.dump(final_models[BEST], MODEL_DIR / "model1_baseline.joblib")

pd.DataFrame({TARGET_DATE_COL: va[TARGET_DATE_COL], "y_true": y_va,
              "y_pred_model1": preds[BEST], "y_pred_naif": h_va}) \
  .to_csv(REPORT_DIR / "pred_val_model1.csv", index=False)

tabel.round(4).to_csv(REPORT_DIR / "val_scores_baseline.csv")

konfig = {"model_terpilih": BEST,
          "alpha": None if np.isnan(hasil[BEST]["alpha"]) else float(hasil[BEST]["alpha"]),
          "target": "selisih" if "selisih" in BEST else "level",
          "fitur": FEATURE_COLS, "seed": SEED, "n_splits_cv": N_SPLITS,
          "metrik_val": {k: float(hasil[BEST][k]) for k in ("MAE", "RMSE", "MAPE_%")}}
with open(REPORT_DIR / "model1_config.json", "w", encoding="utf-8") as f:
    json.dump(konfig, f, ensure_ascii=False, indent=2)
print("Tersimpan: models/model1_baseline.joblib, reports/pred_val_model1.csv, val_scores_baseline.csv, model1_config.json")

## Checklist Model 1 (Minggu 3)
- [ ] Alasan pemilihan model menyebut karakteristik data (autokorelasi tinggi, lag berkorelasi, harga persisten, lonjakan)
- [ ] Asumsi dan inductive bias dijelaskan (Bab 6.1.2)
- [ ] Hyperparameter dan hasil pada **validation set** tercatat, test set tidak dipakai
- [ ] Semua bagian bertanda `[ ... ]` sudah diisi
- [ ] Notebook dijalankan ulang dari awal sampai akhir tanpa error
- [ ] Commit: `git add notebooks/03_model_design_baseline_ml.ipynb`, lalu pull request untuk di-review
- [ ] Skor baseline dibagikan ke pemilik Model 2 (`reports/val_scores_baseline.csv`) untuk perbandingan awal
- [ ] Ringkasan untuk asistensi **Checkpoint 2** disiapkan